# Audio GRPO on Qwen3-ASR-1.7B

Most RL tutorials train a text model. This one trains an **audio** model —
[Qwen3-ASR-1.7B](https://huggingface.co/Qwen/Qwen3-ASR-1.7B), a speech
recognizer — end-to-end with GRPO, and the only thing you write is the reward.

The loop:

1. Load LibriSpeech speech clips with `LibriSpeechASRDataset` (a
   `MultimodalDataset` with `modality="audio"`).
2. slime serves Qwen3-ASR on SGLang's `/v1/audio/transcriptions` endpoint and
   the gym's audio-transcription rollout posts each clip, collecting the
   transcript.
3. Your `wer_reward` scores each transcript as **−WER** (word error rate)
   against the reference text.
4. That reward drives a GRPO update through slime/Megatron.

Everything audio-specific — serving on the transcription endpoint, padded
(bshd) batches the bridge requires, the audio I/O dependencies, the upstream
compat shims, and the trained-model HF export — is handled by the gym when you
pick `model=Qwen3ASR()` and `recipe=Qwen3ASR_Recipe(...)`. You just bring the
reward.

## Prerequisites

This tutorial requires a Modal Secret named `huggingface-secret` containing your
`HF_TOKEN`. Create one at [modal.com/secrets](https://modal.com/secrets) if you
haven't already — the cell below fails fast with instructions otherwise.

> **Note:** you do **not** need to attach a GPU to this notebook. All training and
> serving happens on Modal-managed GPU workers spun up by the SDK — the notebook
> itself only needs to issue API calls.

In [ ]:
import modal

try:
    modal.Secret.from_name("huggingface-secret").hydrate()
except modal.exception.NotFoundError as e:
    raise RuntimeError(
        "Missing Modal Secret 'huggingface-secret'. Create one at "
        "https://modal.com/secrets with an HF_TOKEN entry, then re-run."
    ) from e

In [ ]:
%uv pip install -q git+https://github.com/modal-projects/training-gym.git@main

In [ ]:
from modal_training_gym import (
    LibriSpeechASRDataset,
    Qwen3ASR,
    Qwen3ASR_Recipe,
    TrainConfig,
    WandbConfig,
)

## Load LibriSpeech audio

`LibriSpeechASRDataset` pulls a few clips from the standard LibriSpeech dummy
set. Each row is a prompt with an `<audio>` placeholder, the clip itself (as a
base64 `data:audio/wav` URI), and the reference transcript as the label. As a
`MultimodalDataset` it tells slime to forward the audio column to the rollout —
the same passthrough images and video use.

In [ ]:
dataset = LibriSpeechASRDataset(n_rows=8)

Let's look at a row — text prompt, an audio data-URI, and the reference label.

In [ ]:
row = dataset.load()[0]
print("prompt:", row["prompt"])
print("audio: ", row["audios"][0][:48], "...")
print("label: ", row["label"])

## Define the reward

This is the one task-specific piece. slime calls the reward once per rollout
sample with a `Sample` carrying `.response` (the transcript the model produced)
and `.label` (the reference). We score it as **negative word error rate** so
that lower WER → higher reward, and GRPO pushes the model toward more accurate
transcripts. (`jiwer` is installed for you with the model.)

Qwen3-ASR is already near-perfect on clean LibriSpeech, so the `Qwen3ASR_Recipe`
defaults sample many transcripts per clip at temperature 1.0 — that's what gives
the GRPO group enough within-group WER variance to produce a non-zero gradient.

In [ ]:
async def wer_reward(args, sample, **kwargs) -> float:
    import jiwer

    response = (getattr(sample, "response", "") or "").lower().strip()
    reference = (getattr(sample, "label", "") or "").lower().strip()
    if not reference:
        return 0.0
    return -float(jiwer.wer(reference, response))

## Train

`Qwen3ASR_Recipe` carries the ASR-specific defaults — the transcription
rollout, padded (bshd) batches, the lighter SGLang memory fraction, and the
many-samples/high-temperature settings that surface reward variance — so the
recipe you write only sets the reward and (optionally) W&B logging. It defaults
to a 2×H100 single node; pass `actor_num_gpus_per_node=8` (and a larger
`num_rollout`) to use a full node.

`TrainConfig.train()` builds the Modal app, runs GRPO, and exports the trained
model as a standard HuggingFace checkpoint (audio tower included).

In [ ]:
training_run = TrainConfig(
    model=Qwen3ASR(),
    dataset=dataset,
    recipe=Qwen3ASR_Recipe(
        custom_rm_function=wer_reward,
        wandb=WandbConfig(
            project="qwen3-asr-rl",
            group="gym-demo",
            exp_name="qwen3-asr-grpo-audio-demo",
        ),
    ),
)
print("Starting training...")
train_result = training_run.train()
print(f"Training run id: {train_result.training_run_id}")